# Experiment 1: Data Collection

| Field | Value |
|---|---|
| **Name** | Francis Reuben R |
| **USN** | 1RUA25CSE0168 |
| **Section** | G |
| **Course** | Data Science Laboratory |
| **Semester** | 3 |
| **Experiment** | 1 |

## Bengaluru weather and air quality (Open-Meteo APIs)


### Objective

Collect hourly weather and air-quality records for five Bengaluru locations and save them as CSV files.

| Dataset | API | Window | Fields |
|---|---|---|---|
| Weather | `https://api.open-meteo.com/v1/forecast` | past 21 days, hourly | temperature, relative humidity |
| Air quality | `https://air-quality-api.open-meteo.com/v1/air-quality` | past 30 days, hourly | PM10, PM2.5, CO |

Locations: City Center, Whitefield, Electronic City, Jayanagar, Yeshwanthpur.

### Why an API instead of scraping

Open-Meteo publishes a documented JSON API with no key. If an API exists, use it. Scrape only when there is no API and the site allows it.


### Setup


In [1]:
import time
import requests
import pandas as pd

STUDENT = {
    "name": "Francis Reuben R",
    "usn": "1RUA25CSE0168",
    "section": "G",
}

HEADERS = {
    "User-Agent": "DS-Lab-Exp1/1.0 (Francis Reuben R; USN 1RUA25CSE0168; Section G; academic)"
}

BENGALURU_LOCATIONS = [
    {"name": "City Center", "latitude": 12.9716, "longitude": 77.5946},
    {"name": "Whitefield", "latitude": 12.9698, "longitude": 77.7500},
    {"name": "Electronic City", "latitude": 12.8409, "longitude": 77.6749},
    {"name": "Jayanagar", "latitude": 12.9244, "longitude": 77.5830},
    {"name": "Yeshwanthpur", "latitude": 13.0242, "longitude": 77.5516},
]

def get_json(url, params, retries=5):
    last = None
    for attempt in range(1, retries + 1):
        last = requests.get(url, params=params, headers=HEADERS, timeout=30)
        if last.status_code in (429, 500, 502, 503, 504):
            wait = 1.5 * attempt
            print(f"  {last.status_code} from {url.split('/')[2]} (attempt {attempt}/{retries}), retrying in {wait:.1f}s")
            time.sleep(wait)
            continue
        last.raise_for_status()
        return last.json()
    last.raise_for_status()


print(STUDENT)
print("Locations:", [loc["name"] for loc in BENGALURU_LOCATIONS])


{'name': 'Francis Reuben R', 'usn': '1RUA25CSE0168', 'section': 'G'}
Locations: ['City Center', 'Whitefield', 'Electronic City', 'Jayanagar', 'Yeshwanthpur']


### Weather collection

For each location: build the Open-Meteo URL, send GET, parse hourly `temperature_2m` and `relative_humidity_2m`, append rows, then save `bengaluru_weather.csv`.


In [2]:
def collect_bengaluru_weather():
    api_url = "https://api.open-meteo.com/v1/forecast"
    rows = []

    for location in BENGALURU_LOCATIONS:
        print(f"Fetching weather for {location['name']}...")
        params = {
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "hourly": "temperature_2m,relative_humidity_2m",
            "timezone": "Asia/Kolkata",
            "past_days": 21,
        }
        hourly = get_json(api_url, params)["hourly"]
        for timestamp, temp, humidity in zip(
            hourly["time"],
            hourly["temperature_2m"],
            hourly["relative_humidity_2m"],
        ):
            rows.append({
                "location": location["name"],
                "latitude": location["latitude"],
                "longitude": location["longitude"],
                "timestamp": timestamp,
                "temperature_c": temp,
                "relative_humidity_percent": humidity,
            })
        time.sleep(0.4)

    return pd.DataFrame(rows)


weather_df = collect_bengaluru_weather()
print("Weather rows:", len(weather_df))
weather_df.head()


Fetching weather for City Center...


Fetching weather for Whitefield...


Fetching weather for Electronic City...


Fetching weather for Jayanagar...


Fetching weather for Yeshwanthpur...


Weather rows: 3360


,location,latitude,longitude,timestamp,temperature_c,relative_humidity_percent
0,City Center,12.9716,77.5946,2026-07-30T00:00,22.1,88
1,City Center,12.9716,77.5946,2026-07-30T01:00,21.9,90
2,City Center,12.9716,77.5946,2026-07-30T02:00,21.5,88
3,City Center,12.9716,77.5946,2026-07-30T03:00,21.2,90
4,City Center,12.9716,77.5946,2026-07-30T04:00,21.0,90


In [3]:
weather_path = "bengaluru_weather.csv"
weather_df.to_csv(weather_path, index=False, encoding="utf-8")
print("Saved", weather_path)
print(weather_df.groupby("location").size())
weather_df.describe(include="all")


Saved bengaluru_weather.csv
location
City Center        672
Electronic City    672
Jayanagar          672
Whitefield         672
Yeshwanthpur       672
dtype: int64


,location,latitude,longitude,timestamp,temperature_c,relative_humidity_percent
count,3360,3360.000000,3360.000000,3360,3360.000000,3360.000000
unique,5,NaN,NaN,672,NaN,NaN
top,City Center,NaN,NaN,2026-07-30T00:00,NaN,NaN
freq,672,NaN,NaN,5,NaN,NaN
mean,NaN,12.946180,77.630820,NaN,23.606131,78.188095
std,NaN,0.061409,0.072151,NaN,2.741096,14.065821
min,NaN,12.840900,77.551600,NaN,18.700000,41.000000
25%,NaN,12.924400,77.583000,NaN,21.300000,66.000000
50%,NaN,12.969800,77.594600,NaN,22.900000,82.000000
75%,NaN,12.971600,77.674900,NaN,25.800000,90.000000


### Air quality collection

Same five locations. Open-Meteo Air Quality API, past 30 days, hourly PM10, PM2.5, and carbon monoxide. Save `bengaluru_air_quality.csv`.


In [4]:
def collect_bengaluru_air_quality():
    api_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    rows = []

    for location in BENGALURU_LOCATIONS:
        print(f"Fetching air quality for {location['name']}...")
        params = {
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "hourly": "pm10,pm2_5,carbon_monoxide",
            "timezone": "Asia/Kolkata",
            "past_days": 30,
        }
        hourly = get_json(api_url, params)["hourly"]
        for timestamp, pm10, pm25, co in zip(
            hourly["time"],
            hourly["pm10"],
            hourly["pm2_5"],
            hourly["carbon_monoxide"],
        ):
            rows.append({
                "location": location["name"],
                "latitude": location["latitude"],
                "longitude": location["longitude"],
                "timestamp": timestamp,
                "pm10_ug_m3": pm10,
                "pm2_5_ug_m3": pm25,
                "carbon_monoxide_ug_m3": co,
            })
        time.sleep(0.4)

    return pd.DataFrame(rows)


air_df = collect_bengaluru_air_quality()
print("Air quality rows:", len(air_df))
air_df.head()


Fetching air quality for City Center...


Fetching air quality for Whitefield...


Fetching air quality for Electronic City...


Fetching air quality for Jayanagar...


Fetching air quality for Yeshwanthpur...


Air quality rows: 4200


,location,latitude,longitude,timestamp,pm10_ug_m3,pm2_5_ug_m3,carbon_monoxide_ug_m3
0,City Center,12.9716,77.5946,2026-07-21T00:00,5.6,4.5,233.0
1,City Center,12.9716,77.5946,2026-07-21T01:00,4.7,3.8,186.0
2,City Center,12.9716,77.5946,2026-07-21T02:00,4.6,3.6,158.0
3,City Center,12.9716,77.5946,2026-07-21T03:00,5.0,3.7,148.0
4,City Center,12.9716,77.5946,2026-07-21T04:00,5.6,4.0,155.0


In [5]:
air_path = "bengaluru_air_quality.csv"
air_df.to_csv(air_path, index=False, encoding="utf-8")
print("Saved", air_path)
print(air_df.groupby("location").size())
print("\nMean PM2.5 by location (ug/m3):")
print(air_df.groupby("location")["pm2_5_ug_m3"].mean().round(2))


Saved bengaluru_air_quality.csv
location
City Center        840
Electronic City    840
Jayanagar          840
Whitefield         840
Yeshwanthpur       840
dtype: int64

Mean PM2.5 by location (ug/m3):
location
City Center        6.93
Electronic City    6.93
Jayanagar          6.93
Whitefield         6.93
Yeshwanthpur       6.75
Name: pm2_5_ug_m3, dtype: float64


### Result

Weather and air-quality CSVs are written next to this notebook. Both datasets include the location name so later labs can group, plot, or model by neighbourhood.

### Conclusion

Hourly Bengaluru weather (21 days) and air quality (30 days) were collected from Open-Meteo for five locations and stored as structured CSV files. The APIs returned JSON, which was flattened into pandas DataFrames. A short delay was used between requests.
